# 03 · Streamlit: from charts to a dashboard

**Goal:** turn a notebook analysis into a small web app with widgets, caching, layout and
chart selections, which colleagues can use without touching code.

### The one idea to remember

> A Streamlit app is a Python **script that reruns from top to bottom** every time the user
> touches a widget. A widget call like `st.slider(...)` draws the widget *and returns its current value*.

So there are no callbacks to wire up: you write ordinary top-to-bottom pandas and plotting code,
and Streamlit redraws the page.

### How we'll work

Streamlit apps are `.py` files run from a **terminal**, not notebook cells. In this notebook each
`%%writefile` cell saves a version of the app into `my_apps/`. Then, in a terminal opened in this folder:

```bash
streamlit run my_apps/gapminder_v1.py
```
**OR ON JUPYTERHUB**
```bash
streamlit run my_apps/gapminder_v1.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false
```
And access the app via:
`https://<hub_url>/user/<username>/proxy/8501/` [and **yes**, keep the last `/`]


Leave it running. When you overwrite the file, the browser shows **"Source file changed"**:
click **Always rerun** and it will refresh on every save.

In [ ]:
import os
os.makedirs("my_apps", exist_ok=True)

## Version 1: a chart and a table

`st.title`, `st.plotly_chart`, `st.dataframe`. (`st.altair_chart` works the same way for Altair.)

In [ ]:
%%writefile my_apps/gapminder_v1.py
import plotly.express as px
import streamlit as st

st.title("Wealth and health of nations")

gap = px.data.gapminder()
g07 = gap.query("year == 2007")

fig = px.scatter(g07, x="gdpPercap", y="lifeExp", size="pop", color="continent",
                 hover_name="country", log_x=True, size_max=55)
st.plotly_chart(fig)

st.dataframe(g07)

## Version 2: widgets

Put widgets in the sidebar with `with st.sidebar:`. Each one returns a value you use in ordinary pandas.
Move the slider and watch the terminal: the whole script reruns.

In [ ]:
%%writefile my_apps/gapminder_v2.py
import plotly.express as px
import streamlit as st

st.title("Wealth and health of nations")

gap = px.data.gapminder()

# --- Widgets: each returns its current value ---------------------------
with st.sidebar:
    st.header("Filters")
    year = st.slider("Year", min_value=1952, max_value=2007, value=2007, step=5)
    continents = st.multiselect("Continents", sorted(gap.continent.unique()),
                                default=sorted(gap.continent.unique()))
    log_x = st.checkbox("Log scale for GDP", value=True)

# --- Plain pandas filtering --------------------------------------------
df = gap[(gap.year == year) & (gap.continent.isin(continents))]

fig = px.scatter(df, x="gdpPercap", y="lifeExp", size="pop", color="continent",
                 hover_name="country", log_x=log_x, size_max=55,
                 range_y=[25, 90])
st.plotly_chart(fig)

st.write(f"Showing {len(df)} countries in {year}.")

## Version 3: caching and layout

Because the script reruns on every click, slow steps (loading files, querying databases, fitting
models) must be cached. `@st.cache_data` stores a function's return value keyed on its arguments.

Layout tools: `st.columns`, `st.tabs`, `st.expander`, `st.metric`, and `layout="wide"`.

**Try it:** comment out `@st.cache_data` and move the slider. Every rerun now waits two seconds.

In [ ]:
%%writefile my_apps/gapminder_v3.py
import time

import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Gapminder explorer", layout="wide")


@st.cache_data  # runs once; later reruns reuse the result
def load_data():
    time.sleep(2)  # pretend this is a slow database query
    return px.data.gapminder()


gap = load_data()

with st.sidebar:
    st.header("Filters")
    year = st.slider("Year", 1952, 2007, 2007, step=5)
    continents = st.multiselect("Continents", sorted(gap.continent.unique()),
                                default=sorted(gap.continent.unique()))

df = gap[(gap.year == year) & (gap.continent.isin(continents))]
prev = gap[(gap.year == year - 5) & (gap.continent.isin(continents))]

st.title("Gapminder explorer")

# --- A row of headline numbers ------------------------------------------
c1, c2, c3 = st.columns(3)
c1.metric("Countries", len(df))
c2.metric("Median life expectancy", f"{df.lifeExp.median():.1f} yrs",
          delta=f"{df.lifeExp.median() - prev.lifeExp.median():+.1f} vs 5 yrs earlier"
          if len(prev) else None)
c3.metric("Total population", f"{df['pop'].sum() / 1e9:.2f} bn")

# --- Tabs keep the page tidy --------------------------------------------
tab_chart, tab_table = st.tabs(["Chart", "Data"])
with tab_chart:
    fig = px.scatter(df, x="gdpPercap", y="lifeExp", size="pop", color="continent",
                     hover_name="country", log_x=True, size_max=55, range_y=[25, 90])
    st.plotly_chart(fig, width="stretch")
with tab_table:
    st.dataframe(df, hide_index=True, width="stretch")
    st.download_button("Download CSV", df.to_csv(index=False),
                       file_name=f"gapminder_{year}.csv", mime="text/csv")

## Version 4: the chart becomes an input

With `on_select="rerun"`, selecting points on a Plotly (or Altair) chart reruns the script and
returns the selection, so the chart can drive the rest of the page. Here, lasso a few countries
to see their history.

`custom_data=["country"]` puts the country name in each selected point so we can read it back.

In [ ]:
%%writefile my_apps/gapminder_app.py
"""Gapminder explorer: a complete Streamlit example.

Run with:  streamlit run gapminder_app.py
"""
import plotly.express as px
import streamlit as st

st.set_page_config(page_title="Gapminder explorer", layout="wide")


@st.cache_data
def load_data():
    return px.data.gapminder()

gap = load_data()

with st.sidebar:
    st.header("Filters")
    year = st.slider("Year", 1952, 2007, 2007, step=5)
    continents = st.multiselect("Continents", sorted(gap.continent.unique()),
                                default=sorted(gap.continent.unique()))

df = gap[(gap.year == year) & (gap.continent.isin(continents))]

st.title("Gapminder explorer")
st.caption("Select countries on the scatter plot (click, or box/lasso select) "
           "to see their history on the right.")

c1, c2, c3 = st.columns(3)
c1.metric("Countries", len(df))
c2.metric("Median life expectancy", f"{df.lifeExp.median():.1f} yrs" if len(df) else "–")
c3.metric("Total population", f"{df['pop'].sum() / 1e9:.2f} bn")

left, right = st.columns([3, 2])

with left:
    fig = px.scatter(df, x="gdpPercap", y="lifeExp", size="pop", color="continent",
                     hover_name="country", custom_data=["country"],
                     log_x=True, size_max=55, range_y=[25, 90],
                     labels={"gdpPercap": "GDP per capita", "lifeExp": "Life expectancy"})
    # on_select="rerun": selecting points reruns the script and returns the selection
    event = st.plotly_chart(fig, width="stretch", key="scatter",
                            on_select="rerun", selection_mode=("points", "box", "lasso"))

selected = [p["customdata"][0] for p in event.selection.points]

with right:
    if selected:
        hist = gap[gap.country.isin(selected)]
        st.plotly_chart(px.line(hist, x="year", y="lifeExp", color="country",
                                markers=True, range_y=[25, 90],
                                title="Life expectancy over time"),
                        width="stretch")
    else:
        st.info("Nothing selected yet: click or lasso some countries.")

with st.expander("Show data"):
    shown = df[df.country.isin(selected)] if selected else df
    st.dataframe(shown, hide_index=True, width="stretch")
    st.download_button("Download CSV", shown.to_csv(index=False),
                       file_name=f"gapminder_{year}.csv", mime="text/csv")

## Other building blocks worth knowing

| Need | Use |
| --- | --- |
| Remember values between reruns (a counter, a list of notes) | `st.session_state` |
| Rerun only part of the page | `@st.fragment` |
| Group widgets and submit together | `with st.form("key"):` + `st.form_submit_button` |
| Several pages | a `pages/` folder, or `st.navigation` |
| Upload a user's file | `st.file_uploader` |
| Cache a model or database connection (not data) | `@st.cache_resource` |
| Passwords and API keys | `.streamlit/secrets.toml` and `st.secrets` (never commit it) |

### Common gotchas

- **Everything reruns.** Keep expensive work behind `@st.cache_data`.
- **Don't mutate cached objects.** `st.cache_data` returns a copy each time; mutate that copy, not
  something global.
- **Two identical widgets** (same label, same options) need distinct `key=` arguments.

---

### ✏️ Exercise: Build a tips dashboard

The cell below writes a **starter app** with TODOs into `my_apps/tips_app.py`.

1. Run the cell, then `streamlit run my_apps/tips_app.py` in your terminal.
2. Open `my_apps/tips_app.py` in an editor (JupyterLab's editor is fine) and work through the TODOs,
   saving as you go.

The finished version is in `apps/tips_app_solution.py`. Try not to peek until you're stuck!

In [ ]:
%%writefile my_apps/tips_app.py
"""Exercise: a tips dashboard.  Run with:  streamlit run tips_app.py

Work through the TODOs. Save the file and the browser offers to rerun
(click "Always rerun" so it updates on every save).
"""
import plotly.express as px
import streamlit as st

st.title("Restaurant tips")

# TODO 1: wrap the data loading in a function decorated with @st.cache_data
tips = px.data.tips()
tips["tip_pct"] = tips.tip / tips.total_bill * 100

# TODO 2: in the sidebar, add
#   - a multiselect for `day` (default: all days)
#   - a radio button for `time` with options "Both", "Lunch", "Dinner"
#   - a slider for party size, as a range (hint: value=(1, 6) gives a range slider)
df = tips

# TODO 3: show three st.metric values in st.columns:
#   number of bills, mean tip %, mean bill

# TODO 4: a Plotly scatter of total_bill vs tip, colored by day
st.dataframe(df)

# TODO 5 (stretch): add a download button for the filtered data,
#   and a second tab with a box plot of tip_pct by day

### Stretch goals

- Swap the Plotly scatter for an Altair one with a brush, and pass `on_select="rerun"` to
  `st.altair_chart` so the brushed bills appear in the table.
- Add `st.session_state` to keep a running list of "favorite" bills the user has clicked.